# Theme-weighted grid fill

Setting a theme used to change only **one** entry: the code picked a single seeded theme entry and filled the rest of the grid with no theme awareness, so "food" gave you LEMON plus four unrelated words. This notebook walks through the change that lets the theme bias the **whole** grid: score every word against the theme, and fold that similarity into the backtracking filler's candidate weights.

The fill algorithm itself is covered in `03_algorithm_comparison.ipynb`; here we look only at the theme layer on top of it. It makes a single theme-embedding call; everything else is local.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys, logging, random
import pandas as pd

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv; load_dotenv() # OPENAI_API_KEY for the theme embedding

logging.getLogger().setLevel(logging.ERROR)

In [ ]:
from src.gridgpt.word_database_manager import WordDatabaseManager
from src.gridgpt.template_manager import select_template
from src.gridgpt.theme_manager import ThemeManager
from src.gridgpt.crossword_generator import (
    generate_themed_crossword, normalized_themeness,
    DEFAULT_THEME_BOOST, DEFAULT_SIM_LOW, DEFAULT_SIM_HIGH,
)

word_db = WordDatabaseManager()
theme = 'space'
template = select_template(template_id='5x5_blocked_corners')
DEFAULT_THEME_BOOST, DEFAULT_SIM_LOW, DEFAULT_SIM_HIGH

## 1. Score every word against the theme

`ThemeManager.score_all_words()` embeds the theme once and takes the cosine similarity against the precomputed word-embedding matrix, giving a score for every word in the database. `prepare_theme()` returns both the seeded entry and that full similarity map from a **single** embedding call.

In [ ]:
theme_manager = ThemeManager(theme, word_db)
seed_entry, similarities = theme_manager.prepare_theme(
    threshold=0.35, min_chars=5, max_chars=5, min_frequency=1,
)
print('seeded entry:', seed_entry, '   scored', len(similarities), 'words')

ranked = sorted(similarities.items(), key=lambda kv: kv[1], reverse=True)
pd.DataFrame(ranked[:15], columns=['word', 'cosine similarity'])

## 2. Turn similarity into a selection weight

Raw cosine is clipped to a `[sim_low, sim_high]` band to get a 0..1 **themeness**, then blended multiplicatively with the word's frequency:

`weight = frequency * (1 + boost * themeness)`

Frequency stays the safety net (keeps fills sensible) while similarity is a bounded booster. A word at the top of the band gets `1 + boost` times its usual weight.

In [ ]:
sample = [ranked[0], ranked[len(ranked) // 4], ranked[len(ranked) // 2], ranked[-1]]
rows = []
for word, sim in sample:
    themeness = normalized_themeness(sim, DEFAULT_SIM_LOW, DEFAULT_SIM_HIGH)
    rows.append({
        'word': word,
        'cosine': round(sim, 3),
        'themeness': round(themeness, 2),
        'weight multiplier': round(1 + DEFAULT_THEME_BOOST * themeness, 2),
    })
pd.DataFrame(rows)

## 3. Bias the fill

The weight function is handed to the backtracking filler, which tries higher-weight (more on-theme) words first for each slot. Crucially it only changes the **order** candidates are tried, never the candidate set, so the grid stays valid and generation never fails; it just leans on-theme where it can. A filled word counts as "on-theme" once its themeness clears the visible threshold (0.6).

In [ ]:
random.seed(7)
crossword = generate_themed_crossword(
    template, seed_entry, theme_similarities=similarities, word_db_manager=word_db,
)
print('theme:', theme, '  seeded entry:', seed_entry)
print('filled words:', sorted(crossword['filled_slots'].values()))
# generate_themed_crossword records the theme-related entries itself
# (the seed entries plus any filled word over the visible threshold),
# returned as `theme_entries`:
print('seed_entries:', crossword['seed_entries'])
print('theme_entries:', crossword['theme_entries'])

## 4. Measure the lift (weighting off vs on)

A single grid is anecdotal. Averaging over many generations shows the trend: with weighting on, the filled words are more on-theme on average, and a few more of them clear the visible threshold, at no cost to fill success (still 100%).

In [ ]:
def measure(weighting, runs=30):
    random.seed(1)  # same stream for a fair comparison
    themeness_means, visible_counts = [], []
    for _ in range(runs):
        cw = generate_themed_crossword(
            template, seed_entry,
            theme_similarities=similarities if weighting else None,
            word_db_manager=word_db,
        )
        ts = [normalized_themeness(similarities.get(w), DEFAULT_SIM_LOW, DEFAULT_SIM_HIGH)
              for w in cw['filled_slots'].values()]
        themeness_means.append(sum(ts) / len(ts))
        visible_counts.append(sum(1 for t in ts if t >= 0.6))
    return sum(themeness_means) / runs, sum(visible_counts) / runs

rows = []
for label, weighting in [('off', False), ('on', True)]:
    mean_themeness, mean_visible = measure(weighting)
    rows.append({'weighting': label,
                 'mean themeness': round(mean_themeness, 3),
                 'on-theme words / puzzle': round(mean_visible, 1)})
pd.DataFrame(rows)

## 5. Tuning the parameters

The theme behavior has two separable levers:

- **Selection** (`boost`, and the `sim_low` / `sim_high` band) changes *which* words the fill prefers.
- **Labeling** (`visible_threshold`, plus the band) changes only which filled words we *call* on-theme (the `theme_entries` shown to the solver); it does not touch the puzzle.

This section sweeps both across five themes to find sensible values. It makes one embedding call per theme.

### 5.1 Selection: does `boost` help?

For each theme we generate many puzzles across a range of `boost` values and measure the mean raw cosine of the filled words, a band-independent measure of how on-theme the grid actually is.

Note: an indepth evaluation of these tuning parameters can be found in [`evaluation.md`](../docs/evaluation.md), using [`evaluate_themes.py`](../scripts/evaluate_themes.py)

In [ ]:
from scripts.evaluate_themes import prepare_themes, boost_sweep, threshold_analysis

tuning_themes = ['food', 'music', 'planets', 'sports', 'plants']
tuning_templates = [select_template(template_id=tid) for tid in
                    ['5x5_blocked_corners', '5x5_bottom_pillars', '5x5_diagonal_cut']]

# one embedding call per theme -> {theme: (seed entry, {WORD: cosine})}
prepared = prepare_themes(tuning_themes, word_db)
{th: seed for th, (seed, _) in prepared.items()}

In [ ]:
# mean raw cosine of filled words per theme, across boost values (band-independent)
pd.DataFrame(boost_sweep(prepared, tuning_templates, word_db)).set_index('theme')

### 5.2 Labeling: where to set `visible_threshold`?

A filled word is labeled on-theme when its normalized themeness clears `visible_threshold`; since themeness is the raw cosine clipped to `[sim_low, sim_high]`, each threshold maps to a raw-cosine cutoff. `threshold_analysis` reports the average on-theme words per puzzle at each cutoff, plus the words that a threshold of 0.4 surfaces that 0.6 would miss.

In [ ]:
rows, band_words = threshold_analysis(prepared, tuning_templates, word_db)
display(pd.DataFrame(rows).set_index('theme'))  # avg on-theme words / puzzle at each threshold

print('words counted at vt=0.4 but not vt=0.6 (the 0.4-0.6 themeness band):')
for th in tuning_themes:
    print(f'  {th:<9}: {band_words[th][:12]}')